In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import logging

logging.basicConfig(level=logging.INFO)

# === Paths ===
BASE_DIR      = Path(r"D:\Shikha\bias_project")
WORD_LISTS    = BASE_DIR / "word_lists" / "caliskan_2017"
CORPUS_DIR    = BASE_DIR / "corpus"
OUTPUT_DIR    = BASE_DIR / "outputs" / "embeddings" / "word2vec"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Load word lists for WEAT 6 (Career vs Family) ===
# This is the most relevant test for labor force participation

def load_word_list(filepath):
    with open(filepath, 'r') as f:
        return [w.strip().lower() for w in f.readlines() if w.strip()]

# WEAT 6 — target and attribute words
career_terms  = load_word_list(WORD_LISTS / "career_terms_weat6.txt")
family_terms  = load_word_list(WORD_LISTS / "family_terms_weat6.txt")
male_names    = load_word_list(WORD_LISTS / "male_names_weat6.txt")
female_names  = load_word_list(WORD_LISTS / "female_names_weat6.txt")

print("=== WEAT 6 Word Lists ===")
print(f"Career terms  : {career_terms}")
print(f"Family terms  : {family_terms}")
print(f"Male names    : {male_names}")
print(f"Female names  : {female_names}")

=== WEAT 6 Word Lists ===
Career terms  : ['executive', 'management', 'professional', 'corporation', 'salary', 'office', 'business', 'career']
Family terms  : ['home', 'parents', 'children', 'family', 'cousins', 'marriage', 'wedding', 'relatives']
Male names    : ['john', 'paul', 'mike', 'kevin', 'steve', 'greg', 'jeff', 'bill']
Female names  : ['amy', 'joan', 'lisa', 'sarah', 'diana', 'kate', 'ann', 'donna']


In [2]:
# Load all corpus sources and combine into per-state text
# We'll train one Word2Vec model per state

def load_congressional_record():
    path = CORPUS_DIR / "congressional_record" / "2024-01" / "speeches.csv"
    df = pd.read_csv(path)
    print(f"Congressional Record: {len(df)} speeches, columns: {list(df.columns)}")
    return df

def load_chronicling_america():
    path = CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles.csv"
    df = pd.read_csv(path)
    print(f"Chronicling America: {len(df)} pages, columns: {list(df.columns)}")
    return df

def load_gdelt():
    path = CORPUS_DIR / "gdelt" / "2024-01" / "gdelt_articles_with_text.csv"
    df = pd.read_csv(path)
    # Keep only successfully fetched articles
    df = df[df['text'].notna() & (df['text'] != '')]
    print(f"GDELT: {len(df)} articles, columns: {list(df.columns)}")
    return df

def load_reddit():
    posts    = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "posts.csv")
    comments = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "comments.csv")
    print(f"Reddit: {len(posts)} posts, {len(comments)} comments, columns: {list(posts.columns)}")
    return posts, comments

# Load all sources
print("Loading corpora...\n")
cr_df   = load_congressional_record()
ca_df   = load_chronicling_america()
gd_df   = load_gdelt()
rd_posts, rd_comments = load_reddit()

Loading corpora...

Congressional Record: 547 speeches, columns: ['date', 'package_id', 'granule_id', 'granule_class', 'title', 'speaker_name', 'state', 'party', 'chamber', 'bioguide_id', 'speaker_chamber', 'text', 'text_length']
Chronicling America: 1350 pages, columns: ['state', 'state_abbr', 'year', 'item_id', 'title', 'date', 'text_length', 'text']
GDELT: 702 articles, columns: ['article_url', 'state', 'source_domain', 'source_name', 'source_size', 'article_date', 'fetch_status', 'text_length', 'text']
Reddit: 4423 posts, 4563 comments, columns: ['state', 'state_abbr', 'subreddit', 'post_id', 'title', 'selftext', 'score', 'created_utc', 'text_length']


In [3]:
# Repoint Chronicling America to full-OCR file + dedup + filter to full pages
ca_df = pd.read_csv(CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles_full_ocr.csv")
ca_df = ca_df.drop_duplicates('item_id')
ca_df = ca_df[ca_df['text_length'] >= 2000].reset_index(drop=True)
print(f"Chronicling America: {len(ca_df)} unique full-OCR pages (>=2000 chars)")

Chronicling America: 300 unique full-OCR pages (>=2000 chars)


In [4]:
# Combine all sources into a single per-state text dictionary
# state_abbr -> list of tokenized sentences

from gensim.utils import simple_preprocess

import re
def tokenize(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    sentences = re.split(r'[.!?]+|\n+', text)
    result = []
    for sent in sentences:
        toks = simple_preprocess(sent, deacc=True, min_len=2)
        if len(toks) >= 5:
            result.append(toks)
    return result

# Build state -> sentences mapping
state_corpus = {}  # state_abbr -> list of token lists

# 1. Congressional Record (state column has full state name, need abbr)
# Check what format state is in
print("CR state sample:", cr_df['state'].dropna().unique()[:5])

# 2. Chronicling America
print("CA state sample:", ca_df['state_abbr'].dropna().unique()[:5])

# 3. GDELT
print("GDELT state sample:", gd_df['state'].dropna().unique()[:5])

# 4. Reddit
print("Reddit state sample:", rd_posts['state_abbr'].dropna().unique()[:5])

CR state sample: <ArrowStringArray>
['NY', 'KY', 'IL', 'NC', 'IA']
Length: 5, dtype: str
CA state sample: <ArrowStringArray>
['OH', 'WY', 'CA', 'TX', 'IL']
Length: 5, dtype: str
GDELT state sample: <ArrowStringArray>
['CO', 'PA', 'WA', 'ND', 'IA']
Length: 5, dtype: str
Reddit state sample: <ArrowStringArray>
['AL', 'AK', 'AZ', 'AR', 'CA']
Length: 5, dtype: str


In [5]:
# Build per-(state, period) corpus — proposal-correct design
# Periods: hist_1920s (Chronicling America), modern_2020 (Reddit), modern_2024 (CR + GDELT)

state_period_corpus = {}  # (state_abbr, period) -> list of token lists

def add_to_corpus(state_abbr, period, text):
    if not isinstance(text, str) or len(text.strip()) < 50:
        return
    sentences = tokenize(text)
    if not sentences:
        return
    key = (state_abbr, period)
    if key not in state_period_corpus:
        state_period_corpus[key] = []
    state_period_corpus[key].extend(sentences)
    
# 1. Chronicling America → hist_1920s
for _, row in ca_df.iterrows():
    add_to_corpus(str(row['state_abbr']), 'hist_1920s', str(row.get('text', '')))

# 2. Reddit → modern_2020
for _, row in rd_posts.iterrows():
    text = str(row.get('title', '')) + ' ' + str(row.get('selftext', ''))
    add_to_corpus(str(row['state_abbr']), 'modern_2020', text)
for _, row in rd_comments.iterrows():
    add_to_corpus(str(row['state_abbr']), 'modern_2020', str(row.get('body', '')))

# 3. Congressional Record + GDELT → modern_2024
for _, row in cr_df.iterrows():
    add_to_corpus(str(row['state']), 'modern_2024', str(row.get('text', '')))
for _, row in gd_df.iterrows():
    add_to_corpus(str(row['state']), 'modern_2024', str(row.get('text', '')))

# Summary
periods = ['hist_1920s', 'modern_2020', 'modern_2024']
print(f"{'Period':<14} {'State cells':>12} {'Total sentences':>16}")
print("-" * 45)
for p in periods:
    cells = [k for k in state_period_corpus if k[1] == p]
    sents = sum(len(state_period_corpus[k]) for k in cells)
    print(f"{p:<14} {len(cells):>12} {sents:>16,}")

Period          State cells  Total sentences
---------------------------------------------
hist_1920s                9          125,156
modern_2020              50           16,566
modern_2024              52           46,586


In [6]:
GARG_LISTS = BASE_DIR / "word_lists" / "garg_2018"
male_pairs   = load_word_list(GARG_LISTS / "male_pairs.txt")
female_pairs = load_word_list(GARG_LISTS / "female_pairs.txt")

print(f"Male pairs   ({len(male_pairs)}): {male_pairs}")
print(f"Female pairs ({len(female_pairs)}): {female_pairs}")

def cosine_sim(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def get_vector(model, word):
    try:
        return model.wv[word]
    except KeyError:
        return None

Male pairs   (20): ['he', 'son', 'his', 'him', 'father', 'man', 'boy', 'himself', 'male', 'brother', 'sons', 'fathers', 'men', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews']
Female pairs (20): ['she', 'daughter', 'hers', 'her', 'mother', 'woman', 'girl', 'herself', 'female', 'sister', 'daughters', 'mothers', 'women', 'girls', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces']


In [7]:
def weat_effect_size(model, X_words, Y_words, A_words, B_words,
                     min_targets=3, min_attrs=2):
    """Caliskan effect size with relaxed coverage + coverage reporting."""
    vecs_A = [get_vector(model, a) for a in A_words if get_vector(model, a) is not None]
    vecs_B = [get_vector(model, b) for b in B_words if get_vector(model, b) is not None]
    if len(vecs_A) < min_attrs or len(vecs_B) < min_attrs:
        return None, {}

    def s(word):
        w_vec = get_vector(model, word)
        if w_vec is None:
            return None
        return (np.mean([cosine_sim(w_vec, v) for v in vecs_A]) -
                np.mean([cosine_sim(w_vec, v) for v in vecs_B]))

    s_X = [x for x in (s(w) for w in X_words) if x is not None]
    s_Y = [y for y in (s(w) for w in Y_words) if y is not None]

    if len(s_X) < min_targets or len(s_Y) < min_targets:
        return None, {}

    all_s = s_X + s_Y
    std = np.std(all_s, ddof=1)
    if std == 0:
        return None, {}

    coverage = {
        'n_X': len(s_X), 'n_Y': len(s_Y),
        'n_A': len(vecs_A), 'n_B': len(vecs_B)
    }
    return (np.mean(s_X) - np.mean(s_Y)) / std, coverage

# === Multi-seed scoring with coverage reporting ===
SEEDS   = [42, 7, 123, 2024, 99]
EXCLUDE = {'GU', 'PR', 'VI', 'AS', 'MP'}

results = []

print(f"{'State':<7} {'Period':<13} {'Pairs d':>9} {'±std':>7} {'Coverage (M/F/Car/Fam)':>24}")
print("-" * 65)

for (state_abbr, period) in sorted(state_period_corpus.keys()):
    if state_abbr in EXCLUDE:
        continue
    sentences = state_period_corpus[(state_abbr, period)]
    if len(sentences) < 50:
        continue

    pairs_scores, cov = [], {}
    for seed in SEEDS:
        model = Word2Vec(
            sentences=sentences, vector_size=100, window=5,
            min_count=1, workers=1, epochs=20, seed=seed
        )
        d, c = weat_effect_size(model, male_pairs, female_pairs,
                                career_terms, family_terms)
        if d is not None:
            pairs_scores.append(d)
            cov = c

    if len(pairs_scores) >= 3:
        mean_d = np.mean(pairs_scores)
        std_d  = np.std(pairs_scores, ddof=1)
        cov_str = f"{cov['n_X']}/{cov['n_Y']}/{cov['n_A']}/{cov['n_B']}"
        print(f"{state_abbr:<7} {period:<13} {mean_d:>9.3f} {std_d:>7.3f} {cov_str:>24}")
        results.append({
            'state': state_abbr, 'period': period,
            'n_sentences': len(sentences),
            'weat6_pairs_d':   round(mean_d, 4),
            'weat6_pairs_std': round(std_d, 4),
            'cov_male': cov['n_X'], 'cov_female': cov['n_Y'],
            'cov_career': cov['n_A'], 'cov_family': cov['n_B'],
            'n_seeds': len(pairs_scores),
            'method': 'word2vec_pairs_effectsize'
        })

print(f"\nState-period cells scored: {len(results)}")

INFO:gensim.models.word2vec:collecting all words and their counts
INFO:gensim.models.word2vec:PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
INFO:gensim.models.word2vec:collected 1138 word types from a corpus of 3017 raw words and 229 sentences
INFO:gensim.models.word2vec:Creating a fresh vocabulary
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 1138 unique words (100.00% of original 1138, drops 0)', 'datetime': '2026-07-06T01:40:58.078996', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'prepare_vocab'}
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 3017 word corpus (100.00% of original 3017, drops 0)', 'datetime': '2026-07-06T01:40:58.078996', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.

State   Period          Pairs d    ±std   Coverage (M/F/Car/Fam)
-----------------------------------------------------------------


INFO:gensim.models.word2vec:EPOCH 11: training on 3017 raw words (2352 effective words) took 0.0s, 374540 effective words/s
INFO:gensim.models.word2vec:EPOCH 12: training on 3017 raw words (2338 effective words) took 0.0s, 419132 effective words/s
INFO:gensim.models.word2vec:EPOCH 13: training on 3017 raw words (2327 effective words) took 0.0s, 611628 effective words/s
INFO:gensim.models.word2vec:EPOCH 14: training on 3017 raw words (2352 effective words) took 0.0s, 498791 effective words/s
INFO:gensim.models.word2vec:EPOCH 15: training on 3017 raw words (2354 effective words) took 0.0s, 655108 effective words/s
INFO:gensim.models.word2vec:EPOCH 16: training on 3017 raw words (2361 effective words) took 0.0s, 548126 effective words/s
INFO:gensim.models.word2vec:EPOCH 17: training on 3017 raw words (2366 effective words) took 0.0s, 617755 effective words/s
INFO:gensim.models.word2vec:EPOCH 18: training on 3017 raw words (2360 effective words) took 0.0s, 549643 effective words/s
INFO:gen

AK      modern_2024       0.079   0.496                11/11/7/6


INFO:gensim.models.word2vec:EPOCH 19: training on 2793 raw words (2238 effective words) took 0.0s, 674645 effective words/s
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'training on 55860 raw words (44286 effective words) took 0.1s, 299999 effective words/s', 'datetime': '2026-07-06T01:41:03.879337', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'train'}
INFO:gensim.utils:Word2Vec lifecycle event {'params': 'Word2Vec<vocab=1162, vector_size=100, alpha=0.025>', 'datetime': '2026-07-06T01:41:03.880336', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'created'}
INFO:gensim.models.word2vec:collecting all words and their counts
INFO:gensim.models.word2vec:PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
INFO:gensim.models.word2vec:collecte

...

WY      hist_1920s       -0.040   0.072                15/15/8/8


INFO:gensim.models.word2vec:EPOCH 11: training on 5347 raw words (4186 effective words) took 0.0s, 352292 effective words/s
INFO:gensim.models.word2vec:EPOCH 12: training on 5347 raw words (4143 effective words) took 0.0s, 354843 effective words/s
INFO:gensim.models.word2vec:EPOCH 13: training on 5347 raw words (4196 effective words) took 0.0s, 304700 effective words/s
INFO:gensim.models.word2vec:EPOCH 14: training on 5347 raw words (4172 effective words) took 0.0s, 355142 effective words/s
INFO:gensim.models.word2vec:EPOCH 15: training on 5347 raw words (4174 effective words) took 0.0s, 397047 effective words/s
INFO:gensim.models.word2vec:EPOCH 16: training on 5347 raw words (4162 effective words) took 0.0s, 551711 effective words/s
INFO:gensim.models.word2vec:EPOCH 17: training on 5347 raw words (4163 effective words) took 0.0s, 517419 effective words/s
INFO:gensim.models.word2vec:EPOCH 18: training on 5347 raw words (4155 effective words) took 0.0s, 545025 effective words/s
INFO:gen

WY      modern_2020       0.089   0.501                  7/3/3/3


INFO:gensim.models.word2vec:EPOCH 11: training on 5100 raw words (4091 effective words) took 0.0s, 440186 effective words/s
INFO:gensim.models.word2vec:EPOCH 12: training on 5100 raw words (4091 effective words) took 0.0s, 506349 effective words/s
INFO:gensim.models.word2vec:EPOCH 13: training on 5100 raw words (4059 effective words) took 0.0s, 513382 effective words/s
INFO:gensim.models.word2vec:EPOCH 14: training on 5100 raw words (4090 effective words) took 0.0s, 487009 effective words/s
INFO:gensim.models.word2vec:EPOCH 15: training on 5100 raw words (4111 effective words) took 0.0s, 514138 effective words/s
INFO:gensim.models.word2vec:EPOCH 16: training on 5100 raw words (4107 effective words) took 0.0s, 470043 effective words/s
INFO:gensim.models.word2vec:EPOCH 17: training on 5100 raw words (4074 effective words) took 0.0s, 531861 effective words/s
INFO:gensim.models.word2vec:EPOCH 18: training on 5100 raw words (4100 effective words) took 0.0s, 465512 effective words/s
INFO:gen

WY      modern_2024       0.185   0.865                  6/4/5/3

State-period cells scored: 80


In [8]:
# Diagnostic — check WEAT word coverage in a few representative cells
test_cells = [('WY', 'hist_1920s'), ('TX', 'modern_2020'), ('OH', 'modern_2024'), ('CA', 'modern_2020')]

for key in test_cells:
    if key not in state_period_corpus:
        print(f"{key}: NOT IN CORPUS")
        continue
    sentences = state_period_corpus[key]
    vocab = set(w for sent in sentences for w in sent)
    
    print(f"\n=== {key[0]} {key[1]} ({len(sentences)} sents, {len(vocab):,} unique words) ===")
    for name, words in [('career', career_terms), ('family', family_terms),
                        ('m_pairs', male_pairs), ('f_pairs', female_pairs),
                        ('m_names', male_names), ('f_names', female_names)]:
        present = [w for w in words if w in vocab]
        missing = [w for w in words if w not in vocab]
        print(f"  {name:<9}: {len(present)}/{len(words)} present | missing: {missing}")


=== WY hist_1920s (10535 sents, 11,506 unique words) ===
  career   : 8/8 present | missing: []
  family   : 8/8 present | missing: []
  m_pairs  : 15/20 present | missing: ['males', 'uncle', 'uncles', 'nephew', 'nephews']
  f_pairs  : 15/20 present | missing: ['female', 'females', 'sisters', 'aunts', 'nieces']
  m_names  : 4/8 present | missing: ['kevin', 'steve', 'greg', 'jeff']
  f_names  : 2/8 present | missing: ['joan', 'lisa', 'diana', 'kate', 'ann', 'donna']

=== TX modern_2020 (242 sents, 1,157 unique words) ===
  career   : 3/8 present | missing: ['executive', 'management', 'corporation', 'salary', 'career']
  family   : 1/8 present | missing: ['home', 'parents', 'children', 'cousins', 'marriage', 'wedding', 'relatives']
  m_pairs  : 6/20 present | missing: ['son', 'father', 'boy', 'male', 'brother', 'sons', 'fathers', 'men', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews']
  f_pairs  : 1/20 present | missing: ['she', 'daughter', 'hers', 'mother', 'woman', 'girl',

In [9]:
import json

scores_df = pd.DataFrame(results).sort_values(['period', 'state'])

# Quality flag: reliable if seed-noise std < 0.25 and decent gender-word coverage
scores_df['reliable'] = (
    (scores_df['weat6_pairs_std'] < 0.25) &
    (scores_df['cov_male'] >= 5) & (scores_df['cov_female'] >= 4)
)

OUTPUT_SCORES = OUTPUT_DIR / "weat6_scores_word2vec_v2.csv"
scores_df.to_csv(OUTPUT_SCORES, index=False)

# === Gap documentation per period ===
ALL_STATES = set(STATE_TO_REGION := {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN',
    'IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV',
    'NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN',
    'TX','UT','VT','VA','WA','WV','WI','WY'
})

gap_doc = {'method': 'word2vec_pairs_effectsize_multiseed', 'periods': {}}
for period in ['hist_1920s', 'modern_2020', 'modern_2024']:
    scored = set(scores_df[scores_df['period'] == period]['state'])
    # hist_1920s only covers the 9 extracted states
    expected = {'OH','WY','CA','TX','IL','MS','MT','OR','NY'} if period == 'hist_1920s' else ALL_STATES
    gap = sorted(expected - scored)
    reliable = set(scores_df[(scores_df['period']==period) & scores_df['reliable']]['state'])
    gap_doc['periods'][period] = {
        'expected_states': len(expected),
        'scored': len(scored),
        'reliable': len(reliable),
        'gap_states': gap,
        'gap_reason': 'Insufficient WEAT term coverage or <3 valid seeds at demo corpus scale'
    }

gap_doc['mitigation'] = 'Full-scale extraction will increase per-cell corpus 10-50x, resolving coverage gaps and reducing seed noise.'

GAP_FILE = OUTPUT_DIR / "weat6_gap_documentation_v2.json"
with open(GAP_FILE, 'w') as f:
    json.dump(gap_doc, f, indent=2)

# === Report ===
print(f"Total cells scored : {len(scores_df)}")
print(f"Reliable cells     : {scores_df['reliable'].sum()}")
print(f"\nPer period:")
for p, info in gap_doc['periods'].items():
    print(f"  {p:<13}: {info['scored']}/{info['expected_states']} scored, {info['reliable']} reliable, {len(info['gap_states'])} gap")

print(f"\nReliable cells detail:")
print(scores_df[scores_df['reliable']][
    ['state','period','weat6_pairs_d','weat6_pairs_std','n_sentences']
].to_string(index=False))

print(f"\nSaved to : {OUTPUT_SCORES}")



print(f"Gap doc  : {GAP_FILE}")

Total cells scored : 80
Reliable cells     : 20

Per period:
  hist_1920s   : 9/9 scored, 8 reliable, 0 gap
  modern_2020  : 27/50 scored, 1 reliable, 23 gap
  modern_2024  : 44/50 scored, 11 reliable, 6 gap

Reliable cells detail:
state      period  weat6_pairs_d  weat6_pairs_std  n_sentences
   CA  hist_1920s        -0.2975           0.1809         5115
   IL  hist_1920s         0.3152           0.0719        17404
   MS  hist_1920s        -0.5716           0.1301        12968
   NY  hist_1920s         0.3453           0.1173        30126
   OH  hist_1920s        -0.1240           0.2163        16188
   OR  hist_1920s        -0.0442           0.0878        16931
   TX  hist_1920s         0.6166           0.0767        12662
   WY  hist_1920s        -0.0395           0.0724        10535
   MS modern_2020         0.5069           0.1387          411
   CT modern_2024        -0.5886           0.2289          673
   FL modern_2024         0.2378           0.2094         1234
   IA modern

In [10]:
# === Measure 2: Competence–Warmth | Measure 3: Leadership–Subordination ===
# Both use Garg gender pairs as targets, per proposal

# Load attribute lists
competence_terms = load_word_list(GARG_LISTS / "adjectives_intelligencegeneral.txt")
warmth_terms_raw = load_word_list(GARG_LISTS / "adjectives_sensitive.txt")

# Remove competence-overlapping words from warmth list (documented curation)
OVERLAP_REMOVED = [w for w in warmth_terms_raw if w in competence_terms]
warmth_terms = [w for w in warmth_terms_raw if w not in competence_terms]
print(f"Warmth list: removed {len(OVERLAP_REMOVED)} competence-overlapping words: {OVERLAP_REMOVED}")
print(f"Warmth list final: {len(warmth_terms)} words\n")

# Load custom leadership lists
LEAD_DIR = BASE_DIR / "word_lists" / "custom_leadership"
leadership_terms    = load_word_list(LEAD_DIR / "leadership_terms.txt")
subordination_terms = load_word_list(LEAD_DIR / "subordination_terms.txt")
print(f"Leadership terms    : {len(leadership_terms)}")
print(f"Subordination terms : {len(subordination_terms)}\n")

# === Score both measures across all cells ===
SEEDS = [42, 7, 123, 2024, 99]
results_m23 = []

print(f"{'State':<7} {'Period':<13} {'CompWarm d':>11} {'±std':>7} {'LeadSub d':>10} {'±std':>7}")
print("-" * 62)

for (state_abbr, period) in sorted(state_period_corpus.keys()):
    if state_abbr in EXCLUDE:
        continue
    sentences = state_period_corpus[(state_abbr, period)]
    if len(sentences) < 50:
        continue

    cw_scores, ls_scores = [], []
    for seed in SEEDS:
        model = Word2Vec(
            sentences=sentences, vector_size=100, window=5,
            min_count=1, workers=1, epochs=20, seed=seed
        )
        # Measure 2: positive = male more associated with competence
        d_cw, _ = weat_effect_size(model, male_pairs, female_pairs,
                                   competence_terms, warmth_terms)
        # Measure 3: positive = male more associated with leadership
        d_ls, _ = weat_effect_size(model, male_pairs, female_pairs,
                                   leadership_terms, subordination_terms)
        if d_cw is not None:
            cw_scores.append(d_cw)
        if d_ls is not None:
            ls_scores.append(d_ls)

    row = {'state': state_abbr, 'period': period, 'n_sentences': len(sentences)}
    cw_str, ls_str = "   --", "   --"
    
    if len(cw_scores) >= 3:
        row['compwarm_d']   = round(np.mean(cw_scores), 4)
        row['compwarm_std'] = round(np.std(cw_scores, ddof=1), 4)
        cw_str = f"{np.mean(cw_scores):>7.3f} {np.std(cw_scores, ddof=1):>7.3f}"
    if len(ls_scores) >= 3:
        row['leadsub_d']   = round(np.mean(ls_scores), 4)
        row['leadsub_std'] = round(np.std(ls_scores, ddof=1), 4)
        ls_str = f"{np.mean(ls_scores):>7.3f} {np.std(ls_scores, ddof=1):>7.3f}"
    
    if 'compwarm_d' in row or 'leadsub_d' in row:
        print(f"{state_abbr:<7} {period:<13} {cw_str:>19} {ls_str:>18}")
        results_m23.append(row)

print(f"\nCells scored (either measure): {len(results_m23)}")
print(f"Competence–Warmth cells : {sum(1 for r in results_m23 if 'compwarm_d' in r)}")
print(f"Leadership–Subord cells : {sum(1 for r in results_m23 if 'leadsub_d' in r)}")

INFO:gensim.models.word2vec:collecting all words and their counts
INFO:gensim.models.word2vec:PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
INFO:gensim.models.word2vec:collected 1138 word types from a corpus of 3017 raw words and 229 sentences
INFO:gensim.models.word2vec:Creating a fresh vocabulary
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 1138 unique words (100.00% of original 1138, drops 0)', 'datetime': '2026-07-06T01:45:54.623765', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'prepare_vocab'}
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 3017 word corpus (100.00% of original 3017, drops 0)', 'datetime': '2026-07-06T01:45:54.623765', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.

Warmth list: removed 2 competence-overlapping words: ['intelligent', 'brilliant']
Warmth list final: 18 words

Leadership terms    : 13
Subordination terms : 11

State   Period         CompWarm d    ±std  LeadSub d    ±std
--------------------------------------------------------------


INFO:gensim.models.word2vec:EPOCH 3: training on 3017 raw words (2331 effective words) took 0.0s, 491440 effective words/s
INFO:gensim.models.word2vec:EPOCH 4: training on 3017 raw words (2378 effective words) took 0.0s, 383252 effective words/s
INFO:gensim.models.word2vec:EPOCH 5: training on 3017 raw words (2341 effective words) took 0.0s, 442952 effective words/s
INFO:gensim.models.word2vec:EPOCH 6: training on 3017 raw words (2359 effective words) took 0.0s, 505735 effective words/s
INFO:gensim.models.word2vec:EPOCH 7: training on 3017 raw words (2348 effective words) took 0.0s, 779833 effective words/s
INFO:gensim.models.word2vec:EPOCH 8: training on 3017 raw words (2342 effective words) took 0.0s, 588191 effective words/s
INFO:gensim.models.word2vec:EPOCH 9: training on 3017 raw words (2351 effective words) took 0.0s, 770365 effective words/s
INFO:gensim.models.word2vec:EPOCH 10: training on 3017 raw words (2361 effective words) took 0.0s, 790981 effective words/s
INFO:gensim.mod

AK      modern_2024         0.111   0.331                 --


INFO:gensim.utils:Word2Vec lifecycle event {'update': False, 'trim_rule': 'None', 'datetime': '2026-07-06T01:46:01.417112', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'build_vocab'}
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'training model with 1 workers on 1162 vocabulary and 100 features, using sg=0 hs=0 sample=0.001 negative=5 window=5 shrink_windows=True', 'datetime': '2026-07-06T01:46:01.418110', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'train'}
INFO:gensim.models.word2vec:EPOCH 0: training on 2793 raw words (2221 effective words) took 0.0s, 521276 effective words/s
INFO:gensim.models.word2vec:EPOCH 1: training on 2793 raw words (2216 effective words) took 0.0s, 571488 effective words/s
INFO:gensim.models.word2vec:EPOCH 2: training on

...

WV      modern_2024        -0.083   0.372                 --


INFO:gensim.models.word2vec:resetting layer weights
INFO:gensim.utils:Word2Vec lifecycle event {'update': False, 'trim_rule': 'None', 'datetime': '2026-07-06T01:50:38.584932', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'build_vocab'}
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'training model with 1 workers on 11506 vocabulary and 100 features, using sg=0 hs=0 sample=0.001 negative=5 window=5 shrink_windows=True', 'datetime': '2026-07-06T01:50:38.585910', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'train'}
INFO:gensim.models.word2vec:EPOCH 0: training on 64992 raw words (50101 effective words) took 0.1s, 336078 effective words/s
INFO:gensim.models.word2vec:EPOCH 1: training on 64992 raw words (50094 effective words) took 0.1s, 487024 effective

WY      hist_1920s          0.006   0.047     -0.027   0.071


INFO:gensim.models.word2vec:EPOCH 17: training on 5347 raw words (4163 effective words) took 0.0s, 708222 effective words/s
INFO:gensim.models.word2vec:EPOCH 18: training on 5347 raw words (4155 effective words) took 0.0s, 624991 effective words/s
INFO:gensim.models.word2vec:EPOCH 19: training on 5347 raw words (4147 effective words) took 0.0s, 775981 effective words/s
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'training on 106940 raw words (83289 effective words) took 0.2s, 467996 effective words/s', 'datetime': '2026-07-06T01:50:49.401570', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'train'}
INFO:gensim.utils:Word2Vec lifecycle event {'params': 'Word2Vec<vocab=1654, vector_size=100, alpha=0.025>', 'datetime': '2026-07-06T01:50:49.402775', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platfor


Cells scored (either measure): 24
Competence–Warmth cells : 21
Leadership–Subord cells : 11


In [11]:
m23_df = pd.DataFrame(results_m23).sort_values(['period', 'state'])

# Reliability flags per measure
m23_df['compwarm_reliable'] = (m23_df.get('compwarm_std', np.nan) < 0.25)
m23_df['leadsub_reliable']  = (m23_df.get('leadsub_std', np.nan) < 0.25)

OUT_M23 = OUTPUT_DIR / "measures23_scores_word2vec.csv"
m23_df.to_csv(OUT_M23, index=False)

print(f"Cells saved: {len(m23_df)}")
print(f"Reliable Competence–Warmth cells : {m23_df['compwarm_reliable'].sum()}")
print(f"Reliable Leadership–Subord cells : {m23_df['leadsub_reliable'].sum()}")
print(f"Saved to: {OUT_M23}")

# Curation documentation
curation_doc = {
    'measure2_competence_warmth': {
        'competence_list': 'garg_2018/adjectives_intelligencegeneral.txt (28 words)',
        'warmth_list': 'garg_2018/adjectives_sensitive.txt minus overlaps (18 words)',
        'overlap_removed': OVERLAP_REMOVED,
        'targets': 'garg_2018 male_pairs/female_pairs'
    },
    'measure3_leadership_subordination': {
        'leadership_list': 'custom_leadership/leadership_terms.txt (16 words)',
        'subordination_list': 'custom_leadership/subordination_terms.txt (16 words)',
        'curation_doc': 'word_lists/custom_leadership/README.md',
        'targets': 'garg_2018 male_pairs/female_pairs'
    },
    'sign_convention': 'positive = male more associated with first attribute (competence / leadership)'
}
with open(OUTPUT_DIR / "measures23_curation.json", 'w') as f:
    json.dump(curation_doc, f, indent=2)
print(f"Curation doc saved.")

Cells saved: 24
Reliable Competence–Warmth cells : 10
Reliable Leadership–Subord cells : 5
Saved to: D:\Shikha\bias_project\outputs\embeddings\word2vec\measures23_scores_word2vec.csv
Curation doc saved.


In [12]:
from collections import Counter
import pandas as pd

ERA_DIR = BASE_DIR / "word_lists" / "era_names"
  # word_lists/era_names ; if OUTPUT_DIR was reassigned, hardcode the path

# Map corpus periods -> era-list labels we built
PERIOD_TO_ERA = {
    'hist_1920s':   'historical_1920s',
    'modern_2020':  'modern_2024',
    'modern_2024':  'modern_2024',
}

# Load era name lists: era_label -> {'male': set, 'female': set}
era_names = {}
for f in ERA_DIR.glob("*_names_*.txt"):
    sex, _, era = f.stem.split("_", 2)
    era_names.setdefault(era, {})[sex] = set(f.read_text(encoding="utf-8").split())

# Coverage over the SAME tokenized cells Word2Vec scored on
rows = []
for (state, period), sentences in state_period_corpus.items():
    era = PERIOD_TO_ERA.get(period)
    if era not in era_names:
        continue
    counts = Counter(tok for sent in sentences for tok in sent)  # already simple_preprocess'd
    n_tokens = sum(counts.values())
    for sex in ("male", "female"):
        names   = era_names[era][sex]
        present = {n for n in names if counts[n] > 0}
        hits    = sum(counts[n] for n in present)
        rows.append({
            "state": state, "period": period, "sex": sex,
            "names_total":   len(names),
            "names_present": len(present),
            "coverage_pct":  round(100 * len(present) / len(names), 1) if names else 0,
            "name_hits":     hits,
            "corpus_tokens": n_tokens,
            "hits_per_10k":  round(10000 * hits / n_tokens, 2) if n_tokens else 0,
        })

cov = pd.DataFrame(rows)

print("=== Mean coverage by period × sex ===")
print(cov.groupby(["period", "sex"])[["names_present", "coverage_pct", "hits_per_10k"]].mean().round(2))

print("\n=== Cells with ANY name coverage ===")
print(f"{(cov['names_present'] > 0).sum()} of {len(cov)} (state,period,sex) groups have ≥1 name present")

print("\nLowest-coverage cells:")
print(cov.sort_values("coverage_pct").head(10).to_string(index=False))

VAL_DIR = BASE_DIR / "outputs" / "validation"
VAL_DIR.mkdir(parents=True, exist_ok=True)
cov.to_csv(VAL_DIR / "era_name_coverage.csv", index=False)
print(f"\nSaved: {VAL_DIR / 'era_name_coverage.csv'}")

=== Mean coverage by period × sex ===
                    names_present  coverage_pct  hits_per_10k
period      sex                                              
hist_1920s  female          11.33         56.67          3.61
            male            17.56         87.78         16.31
modern_2020 female           0.48          2.40          1.91
            male             1.24          6.20          3.62
modern_2024 female           2.62         13.08          3.51
            male             6.35         31.73         13.11

=== Cells with ANY name coverage ===
154 of 222 (state,period,sex) groups have ≥1 name present

Lowest-coverage cells:
state      period    sex  names_total  names_present  coverage_pct  name_hits  corpus_tokens  hits_per_10k
   CA modern_2020 female           20              0           0.0          0           4923           0.0
   CA modern_2020   male           20              0           0.0          0           4923           0.0
   CO modern_2020 female 

In [13]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"D:\Shikha\bias_project")
EMB = BASE_DIR / "outputs" / "embeddings"

files = {
    "word2vec": EMB / "word2vec" / "weat6_scores_word2vec_v2.csv",
    "glove":    EMB / "glove"    / "weat6_scores_glove.csv",
    "bert":     EMB / "bert"     / "seat_scores_bert.csv",
    "roberta":  EMB / "roberta"  / "seat_scores_roberta.csv",
}

for name, path in files.items():
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{name}  ({path.name}, {len(df)} rows)")
        print("  columns:", list(df.columns))
        print(df.head(2).to_string(index=False))
    else:
        print(f"\n{name}: FILE NOT FOUND at {path}")


word2vec  (weat6_scores_word2vec_v2.csv, 80 rows)
  columns: ['state', 'period', 'n_sentences', 'weat6_pairs_d', 'weat6_pairs_std', 'cov_male', 'cov_female', 'cov_career', 'cov_family', 'n_seeds', 'method', 'reliable']
state     period  n_sentences  weat6_pairs_d  weat6_pairs_std  cov_male  cov_female  cov_career  cov_family  n_seeds                    method  reliable
   CA hist_1920s         5115        -0.2975           0.1809        16          14           4           6        5 word2vec_pairs_effectsize      True
   IL hist_1920s        17404         0.3152           0.0719        16          15           8           7        5 word2vec_pairs_effectsize      True

glove  (weat6_scores_glove.csv, 50 rows)
  columns: ['state', 'text_chars', 'weat6_score', 'method', 'period']
state  text_chars  weat6_score         method     period
   AK      188041       0.1703 glove_weighted mixed_demo
   AL       63233       0.2114 glove_weighted mixed_demo

bert  (seat_scores_bert.csv, 108 rows

In [14]:
import numpy as np, pandas as pd
from gensim.models import Word2Vec

KIN = {"father","brother","uncle","nephew","son","fathers","brothers","uncles",
       "nephews","sons","mother","sister","aunt","niece","daughter","mothers",
       "sisters","aunts","nieces","daughters"}

male_clean   = [w for w in male_pairs   if w not in KIN]
female_clean = [w for w in female_pairs if w not in KIN]
print(f"Full: {len(male_pairs)}m/{len(female_pairs)}f | Stripped: {len(male_clean)}m/{len(female_clean)}f\n")

def mean_cos_to(model, word, attr_words):
    wv = get_vector(model, word)
    if wv is None: return None
    sims = [cosine_sim(wv, get_vector(model, a)) for a in attr_words if get_vector(model, a) is not None]
    return float(np.mean(sims)) if sims else None

ablation_rows, leak_rows = [], []
all_names = [n.lower() for n in male_names + female_names]

for (state_abbr, period) in sorted(state_period_corpus.keys()):
    if state_abbr in EXCLUDE: continue
    sentences = state_period_corpus[(state_abbr, period)]
    if len(sentences) < 50: continue

    d_full_s, d_clean_s, kin_s, name_s = [], [], [], []
    for seed in SEEDS:
        model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=1, workers=1, seed=seed, epochs=5)
        rf = weat_effect_size(model, male_pairs, female_pairs, career_terms, family_terms)
        rc = weat_effect_size(model, male_clean, female_clean, career_terms, family_terms)
        d_full  = rf[0] if isinstance(rf, tuple) else rf
        d_clean = rc[0] if isinstance(rc, tuple) else rc
        if d_full  is not None: d_full_s.append(d_full)
        if d_clean is not None: d_clean_s.append(d_clean)
        ks = [v for v in (mean_cos_to(model, k, family_terms) for k in KIN) if v is not None]
        ns = [v for v in (mean_cos_to(model, n, family_terms) for n in all_names) if v is not None]
        if ks: kin_s.append(np.mean(ks))
        if ns: name_s.append(np.mean(ns))

    if d_full_s and d_clean_s:
        df_, dc_ = float(np.mean(d_full_s)), float(np.mean(d_clean_s))
        ablation_rows.append({"state": state_abbr, "period": period, "d_full": round(df_,4),
                              "d_kin_stripped": round(dc_,4), "abs_change": round(abs(dc_-df_),4)})
    if kin_s and name_s:
        leak_rows.append({"state": state_abbr, "period": period,
                          "kin_to_family": round(float(np.mean(kin_s)),4),
                          "name_to_family": round(float(np.mean(name_s)),4),
                          "leakage_gap": round(float(np.mean(kin_s))-float(np.mean(name_s)),4)})

abla = pd.DataFrame(ablation_rows); leak = pd.DataFrame(leak_rows)
abla.to_csv(OUTPUT_DIR / "kin_ablation.csv", index=False)
leak.to_csv(OUTPUT_DIR / "kin_leakage_probe.csv", index=False)

print("=== PART A: LEAKAGE PROBE ===")
if len(leak):
    print(f"  kin->family : {leak['kin_to_family'].mean():.4f}")
    print(f"  name->family: {leak['name_to_family'].mean():.4f}")
    print(f"  leakage gap : {leak['leakage_gap'].mean():.4f}  (positive = kin closer = leakage)")
print("\n=== PART B: ABLATION (Career-Family d) ===")
if len(abla):
    print(f"  cells          : {len(abla)}")
    print(f"  mean d full    : {abla['d_full'].mean():.4f}")
    print(f"  mean d stripped: {abla['d_kin_stripped'].mean():.4f}")
    print(f"  mean |change|  : {abla['abs_change'].mean():.4f}")
    print(f"  cells >0.10    : {(abla['abs_change']>0.10).sum()}/{len(abla)}")
print("\nDone.")

INFO:gensim.models.word2vec:collecting all words and their counts
INFO:gensim.models.word2vec:PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
INFO:gensim.models.word2vec:collected 1138 word types from a corpus of 3017 raw words and 229 sentences
INFO:gensim.models.word2vec:Creating a fresh vocabulary
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 1138 unique words (100.00% of original 1138, drops 0)', 'datetime': '2026-07-06T01:50:52.083210', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'prepare_vocab'}
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 3017 word corpus (100.00% of original 3017, drops 0)', 'datetime': '2026-07-06T01:50:52.084310', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.

Full: 20m/20f | Stripped: 10m/10f



INFO:gensim.models.word2vec:EPOCH 1: training on 3017 raw words (2335 effective words) took 0.0s, 384647 effective words/s
INFO:gensim.models.word2vec:EPOCH 2: training on 3017 raw words (2336 effective words) took 0.0s, 428994 effective words/s
INFO:gensim.models.word2vec:EPOCH 3: training on 3017 raw words (2331 effective words) took 0.0s, 358483 effective words/s
INFO:gensim.models.word2vec:EPOCH 4: training on 3017 raw words (2378 effective words) took 0.0s, 532802 effective words/s
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'training on 15085 raw words (11736 effective words) took 0.0s, 271005 effective words/s', 'datetime': '2026-07-06T01:50:52.160053', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'train'}
INFO:gensim.utils:Word2Vec lifecycle event {'params': 'Word2Vec<vocab=1138, vector_size=100, alpha=0.025>', 'datetime': '2026-07-06T01:50:52.161403', 'g

=== PART A: LEAKAGE PROBE ===
  kin->family : 0.2465
  name->family: 0.2599
  leakage gap : -0.0134  (positive = kin closer = leakage)

=== PART B: ABLATION (Career-Family d) ===
  cells          : 72
  mean d full    : -0.0021
  mean d stripped: -0.0054
  mean |change|  : 0.1483
  cells >0.10    : 35/72

Done.
